# Sensitivity to the Number of Keywords k

Not reported in the manuscript. Compares a CRS built from k = 5 keywords with one built from k = 10 on the 5,000-document sample `n5000_r1_s42` of notebook 13. Without any API call the notebook rebuilds the sample, writes the dry-run manifest and mapping for the k = 10 extraction, and runs the k = 5 arm (CRS at tau = 0.40, backbones at w >= 20 and w >= 5, Louvain). If a `predictions_k<k>.csv` from `e4_k_sensitivity/run_extraction.py` is present, that arm is added and compared (backbone overlap, NMI and ARI on shared concepts, extra keywords classified as exact or soft restatements or new concepts).

Inputs: `EID_KEYWORDS.xlsx`, the sample and `runs.csv` of notebook 13, `data/insumo_row_to_eid.csv`, `results/e7_extraction_cost/pricing_snapshot.json`, private record file from `FTTS_PRIVATE_DIR`. Outputs in `aditional_experiments/results/e4_k_sensitivity/`. Gate: the k = 5 arm must match the notebook-13 record of the sample exactly.

In [1]:
# ============================================================
# CONFIGURATION AND IMPORTS
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "4"      # several notebooks may run concurrently on one machine

import contextlib, io, itertools, json, sys
from pathlib import Path
from string import Template

import numpy as np
import pandas as pd
import torch
torch.set_num_threads(4)

# The notebook lives in aditional_experiments/ but every path below is relative to the repository
# root; nbconvert and Jupyter start the kernel in the notebook's own folder, so move up one level.
if Path.cwd().name == "aditional_experiments":
    os.chdir(Path.cwd().parent)
print("working directory:", Path.cwd())

sys.path.insert(0, "scripts")
import common as C
from crs_reference import build_backbone, build_crs_for_tau, parse_keywords

SAMPLE = "results/e3_scalability/samples/n5000_r1_s42.csv"
E3_RUNS = "results/e3_scalability/runs.csv"
KEYWORDS = "EID_KEYWORDS.xlsx"
ALIGNMENT = "data/insumo_row_to_eid.csv"
PRICE_FILE = Path("results/e7_extraction_cost/pricing_snapshot.json")
INSUMO = C.PRIVATE_DIR / "corpus_insumo_DEFINITIVO.csv"      # private Scopus records (Elsevier licence)
OUT = Path("aditional_experiments/results/e4_k_sensitivity")
OUT.mkdir(parents=True, exist_ok=True)

# Extraction configuration of run_extraction.py (notebook 1 with the cardinality as a parameter).
K_VALUES = [10]
MODEL_ID = "meta-llama/llama-3.1-8b-instruct"
MAX_ATTEMPTS, SLEEP_BETWEEN_CALLS, MAX_OUTPUT_TOKENS, TEMPERATURE = 4, 0.6, 700, 0.0
SYSTEM_MSG = (
    'Respond ONLY with a valid JSON in the form {"keywords":[...]}. '
    'Do not include explanations, markdown, apologies, or code blocks. '
    'STRICT LANGUAGE RULE: '
    '- The output MUST contain ONLY ENGLISH WORDS. '
    '- Translate foreign terms into English; if translation is impossible, use a English descriptive equivalent. '
    '- Before responding, internally verify that EVERY keyword is fully in English. '
)
PROMPT_TMPL = Template("""
You are an expert assistant in bibliographic analysis.
From the following article record (authors, title, year, journal, abstract, original keywords),
extract EXACTLY $k terms that represent the main concepts of the article.

Rules:
- ALWAYS return: {"keywords": ["term1", "term2", ...]}.
- Output MUST be ONLY in ENGLISH; no other languages under any circumstance.
- Normalize to lowercase, without accents or special characters (ej. "mathematics education").
- Accept short multiword phrases (2–4 words).
- Avoid generic terms such as: "article", "study", "analysis", "research", "work", "keyword".
- Prioritize disciplinary concepts, population, context, method, theory.
- ALWAYS return exactly $k items in the list; if a concept cannot be expressed in English, use the string "null" as a placeholder, which still counts toward the $k terms.

Article text:
$texto

Before responding, SELF-CHECK: "all the keywords are in English without exception?"
""")

W_LEVELS = [20, 5]       # backbone support: the paper's rule and a level adapted to 5,000 documents
THREADS = 4

if not INSUMO.exists():
    raise FileNotFoundError(
        f"private record file not found: {INSUMO}. Set FTTS_PRIVATE_DIR to the directory that holds "
        "corpus_insumo_DEFINITIVO.csv (Scopus records, not redistributed with this repository).")

# Results archived in the repository before this run, kept in memory for the final comparison cell.
archived = {}
for name in ("summary.json", "validation.json", "dry_run_manifest.json"):
    if (OUT / name).exists():
        archived[name] = json.load(open(OUT / name, encoding="utf-8"))
for name in ("arms_comparison.csv", "sample_mapping.csv"):
    if (OUT / name).exists():
        archived[name] = pd.read_csv(OUT / name)
print("archived files available for the final check:", sorted(archived))

C.set_seeds()
T = C.Timer()
meta = C.env_metadata(experiment="E4 k sensitivity (analysis)", llm_calls=0, paid_api_calls=0,
                      inputs={"sample": {"path": SAMPLE, "sha256": C.sha256(SAMPLE)},
                              "keywords": {"path": KEYWORDS, "sha256": C.sha256(KEYWORDS)},
                              "alignment": {"path": ALIGNMENT, "sha256": C.sha256(ALIGNMENT)},
                              "insumo": {"path": str(INSUMO), "sha256": C.sha256(INSUMO), "redistributed": False}})
print(f"tau = {C.TAU_EDGE} | backbone w in {W_LEVELS} | Louvain seed {C.SEED} | soft-match tau {C.TAU_SOFT}")
for k, v in meta["inputs"].items():
    print(f"  {k:10s} sha256={v['sha256']}")

working directory: /home/mat/academic-writing/papers/from_text_to_structure/data_repo


archived files available for the final check: ['arms_comparison.csv', 'dry_run_manifest.json', 'sample_mapping.csv', 'summary.json', 'validation.json']


/home/mat/academic-writing/papers/from_text_to_structure/data_repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tau = 0.4 | backbone w in [20, 5] | Louvain seed 42 | soft-match tau 0.7
  sample     sha256=de282b2618be149ba65ffcfac9b819466a7ac887acf1ed2ce850258b0505495c
  keywords   sha256=a440b7cedabea950ddb0be27610e1a58f747191ae08999801d73994f9d801c8c
  alignment  sha256=9d592a93d35cde582eb914e40334746f14fd13e743d56176cb8668de83707a03
  insumo     sha256=597d4d8d44ddea693d4386798ea0a9d3380d1314782e937843e3184d78e2f274


In [2]:
# ============================================================
# STEP 1: REBUILD THE 5,000-DOCUMENT SAMPLE FROM THE E3 SAMPLING RULE
# ============================================================
kw = C.load_keywords_xlsx(Path(KEYWORDS), notebook_semantics=True)
corpus = kw[kw.keywords.map(len) > 0].reset_index(drop=True)      # notebook 2 drops empty lists
n_total, n_sample, seed = len(corpus), 5000, 42
rng = np.random.default_rng(np.random.SeedSequence([seed, n_sample]))
positions = np.sort(rng.choice(n_total, size=n_sample, replace=False))
rebuilt = pd.DataFrame({"corpus_position": positions,
                        "source_row_abs": corpus.source_row.values[positions],
                        "EID_o_identificador": corpus.eid.values[positions]})

sample = pd.read_csv(SAMPLE)
sample["EID_o_identificador"] = sample["EID_o_identificador"].astype(str)
same_rows = (len(sample) == len(rebuilt)
             and (sample.corpus_position.values == rebuilt.corpus_position.values).all()
             and (sample.source_row_abs.values == rebuilt.source_row_abs.values).all()
             and (sample.EID_o_identificador.values == rebuilt.EID_o_identificador.values).all())
e3 = pd.read_csv(E3_RUNS)
e3_ref = e3[e3.sample_file.str.endswith(Path(SAMPLE).name)]
sha_file = C.sha256(SAMPLE)
sha_ok = len(e3_ref) == 1 and e3_ref.iloc[0].sample_sha256 == sha_file

print(f"analysed corpus: {n_total} documents; sample: {n_sample} documents, seed {seed}")
print(f"rebuilt sample equals the archived E3 sample row by row: {same_rows}")
print(f"SHA-256 of the sample file matches the E3 record ({sha_file[:16]}...): {sha_ok}")
print(f"first sampled corpus positions: {positions[:8].tolist()} ... last: {positions[-3:].tolist()}")
assert same_rows and sha_ok, "the sample could not be reconciled with E3"
T.mark("sample")

analysed corpus: 52946 documents; sample: 5000 documents, seed 42
rebuilt sample equals the archived E3 sample row by row: True
SHA-256 of the sample file matches the E3 record (de282b2618be149b...): True
first sampled corpus positions: [23, 52, 59, 66, 74, 76, 111, 112] ... last: [52923, 52930, 52931]


4.314

In [3]:
# ============================================================
# STEP 2: SAMPLE-TO-RECORD MAPPING AND DRY-RUN MANIFEST (no API call)
# ============================================================
def build_mapping(sample_path: Path, alignment_path: Path, insumo_path: Path) -> pd.DataFrame:
    """Link every sampled EID to the line of the private record file and measure the record length."""
    sample = pd.read_csv(sample_path)
    align = C.load_alignment(alignment_path)
    first_row = dict(zip(align.eid, align.insumo_row))
    sample["insumo_row"] = [first_row.get(str(e)) for e in sample["EID_o_identificador"]]
    missing = sample.insumo_row.isna().sum()
    if missing:
        raise RuntimeError(f"{missing} sample EIDs could not be linked to a record")
    ins = pd.read_csv(insumo_path)["insumo"].astype(str)
    sample["record_chars"] = [len(ins.iloc[int(r)]) for r in sample.insumo_row]
    sample["text_seen"] = [ins.iloc[int(r)][:C.TRUNCATE_CHARS] for r in sample.insumo_row]
    return sample


mapping = build_mapping(Path(SAMPLE), Path(ALIGNMENT), INSUMO)
mapping.drop(columns=["text_seen"]).to_csv(OUT / "sample_mapping.csv", index=False)   # no record text on disk

manifest = {
    "created_utc": C.now_utc(), "model": MODEL_ID, "k_values": K_VALUES, "documents": int(len(mapping)),
    "sample_file": SAMPLE, "sample_sha256": C.sha256(SAMPLE),
    "decoding": {"temperature": TEMPERATURE, "max_tokens": MAX_OUTPUT_TOKENS},
    "retry_policy": {"max_attempts": MAX_ATTEMPTS, "sleep_between_calls_s": SLEEP_BETWEEN_CALLS},
    "truncate_chars": C.TRUNCATE_CHARS, "records_truncated": int((mapping.record_chars > C.TRUNCATE_CHARS).sum()),
    "system_message": SYSTEM_MSG, "prompt_template": PROMPT_TMPL.template,
    "expected_calls": int(len(mapping) * len(K_VALUES)),
    "wall_clock_lower_bound_hours": len(mapping) * len(K_VALUES) * SLEEP_BETWEEN_CALLS / 3600,
}
if PRICE_FILE.exists():
    pr = json.load(open(PRICE_FILE))
    est_in = 640 * len(mapping) * len(K_VALUES)             # mean prompt length measured in E7
    est_out = 27 * len(mapping) * len(K_VALUES) * (max(K_VALUES) / 5)
    manifest["estimated_cost_usd_at_list_price"] = est_in * pr["usd_per_prompt_token"] + est_out * pr["usd_per_completion_token"]
C.write_json(manifest, OUT / "dry_run_manifest.json")
T.mark("dry_run_manifest")

print(json.dumps({k: v for k, v in manifest.items() if k not in ("system_message", "prompt_template")}, indent=1))
print("dry run: no API call performed")
print("\nexample prompt (k = 10) for the first sampled record, text replaced by its length:")
print(PROMPT_TMPL.substitute(k=K_VALUES[0], texto=f"<record of {mapping.record_chars.iloc[0]} characters, cut at {C.TRUNCATE_CHARS}>"))

{
 "created_utc": "2026-09-24T06:55:38.208606+00:00",
 "model": "meta-llama/llama-3.1-8b-instruct",
 "k_values": [
  10
 ],
 "documents": 5000,
 "sample_file": "results/e3_scalability/samples/n5000_r1_s42.csv",
 "sample_sha256": "de282b2618be149ba65ffcfac9b819466a7ac887acf1ed2ce850258b0505495c",
 "decoding": {
  "temperature": 0.0,
  "max_tokens": 700
 },
 "retry_policy": {
  "max_attempts": 4,
  "sleep_between_calls_s": 0.6
 },
 "truncate_chars": 3000,
 "records_truncated": 185,
 "expected_calls": 5000,
 "wall_clock_lower_bound_hours": 0.8333333333333334,
 "estimated_cost_usd_at_list_price": 0.1816
}
dry run: no API call performed

example prompt (k = 10) for the first sampled record, text replaced by its length:

You are an expert assistant in bibliographic analysis.
From the following article record (authors, title, year, journal, abstract, original keywords),
extract EXACTLY 10 terms that represent the main concepts of the article.

Rules:
- ALWAYS return: {"keywords": ["term1", "t

In [4]:
# ============================================================
# STEP 3: ARMS (k = 5 from the published keywords; k = 10 if predictions are present)
# ============================================================
def notebook_clean(lst):
    return sorted(set(str(k).strip().lower() for k in lst if isinstance(k, str) and k.strip()))


def arm_metrics(docs, emb, label):
    """Global CRS (tau = 0.40) plus backbones at every level of W_LEVELS with Louvain (seed 42)."""
    G = build_crs_for_tau(docs, emb, C.TAU_EDGE)
    res = {"arm": label, "documents": len(docs), "keywords_total": int(sum(len(d) for d in docs)),
           "keywords_per_document_mean": float(np.mean([len(d) for d in docs])),
           "vocabulary": len(set(itertools.chain.from_iterable(docs))),
           "null_tokens": int(sum(d.count("null") for d in docs)), **{f"global_{k}": v for k, v in C.graph_summary(G).items()}}
    parts = {}
    for w in W_LEVELS:
        H = build_backbone(G, w)
        s = C.graph_summary(H)
        part, q = C.louvain_partition(H, C.SEED)
        parts[w] = part
        res.update({f"bb{w}_{k}": v for k, v in s.items()})
        res[f"bb{w}_modularity"] = q
        res[f"bb{w}_communities"] = len(set(part.values())) if part else 0
    return res, G, parts


by_eid = dict(zip(kw.eid, kw.keywords))
eids = [str(e) for e in sample["EID_o_identificador"]]
arms = {5: [by_eid[e] for e in eids]}
pred_files = sorted(OUT.glob("predictions_k*.csv"))
for pf in pred_files:
    k = int(pf.stem.split("_k")[1])
    p = pd.read_csv(pf)
    p["EID_o_identificador"] = p["EID_o_identificador"].astype(str)
    p = p.drop_duplicates("EID_o_identificador").set_index("EID_o_identificador")
    arms[k] = [notebook_clean(parse_keywords(p.loc[e, "keywords_llm"])) if e in p.index and isinstance(p.loc[e, "keywords_llm"], str) else [] for e in eids]
    meta.setdefault("prediction_files", {})[str(pf)] = {"sha256": C.sha256(pf), "rows": int(len(p)),
                                                        "status_counts": pd.read_csv(pf)["status"].value_counts().to_dict()}
print("arms available:", {k: f"{sum(1 for d in v if d)} documents with keywords" for k, v in arms.items()})
print("prediction files found:", [p.name for p in pred_files] or "none (k = 10 arm pending)")

vocab = sorted(set(itertools.chain.from_iterable(itertools.chain.from_iterable(arms.values()))))
model = C.load_embedder(threads=THREADS)
# The batch progress bar of sentence-transformers does not render in an executed notebook.
with contextlib.redirect_stderr(io.StringIO()):
    emb = C.embed_unique(model, vocab, batch_size=256)
T.mark("embeddings")
print(f"embedded {len(emb)} unique keywords")

rows, graphs, parts = [], {}, {}
for k, docs in sorted(arms.items()):
    docs_nonempty = [d for d in docs if d]
    r, G, pk = arm_metrics(docs_nonempty, emb, f"k={k}")
    rows.append(r); graphs[k] = G; parts[k] = pk
table = pd.DataFrame(rows)
table.to_csv(OUT / "arms_comparison.csv", index=False)
T.mark("arms")
print("\nARM METRICS")
print(table.set_index("arm").T.to_string())

arms available: {5: '5000 documents with keywords'}
prediction files found: none (k = 10 arm pending)


embedded 10249 unique keywords



ARM METRICS
arm                                  k=5
documents                    5000.000000
keywords_total              24975.000000
keywords_per_document_mean      4.995000
vocabulary                  10249.000000
null_tokens                    18.000000
global_nodes                10249.000000
global_edges                13247.000000
global_density                  0.000252
global_components            3775.000000
global_lcc_nodes             5692.000000
global_lcc_fraction             0.555371
global_lcc_edges            12353.000000
global_isolated_nodes        3234.000000
bb20_nodes                     31.000000
bb20_edges                     35.000000
bb20_density                    0.075269
bb20_components                 1.000000
bb20_lcc_nodes                 31.000000
bb20_lcc_fraction               1.000000
bb20_lcc_edges                 35.000000
bb20_isolated_nodes             0.000000
bb20_modularity                 0.340116
bb20_communities                3.000000
bb5

In [5]:
# ============================================================
# REPRODUCTION GATE: the k = 5 arm must coincide with the E3 run on the same sample
# ============================================================
k5 = table[table.arm == "k=5"].iloc[0]
checks = {}
if len(e3_ref) == 1:
    ref = e3_ref.iloc[0]
    for ours, theirs in [("global_nodes", "nodes_full_graph"), ("global_edges", "edges_full_graph"),
                         ("global_components", "n_components_full_graph"), ("global_lcc_nodes", "lcc_nodes"),
                         ("bb20_nodes", "backbone_nodes"), ("bb20_edges", "backbone_edges"),
                         ("bb20_communities", "n_communities")]:
        checks[ours] = {"observed": int(k5[ours]), "e3_reference": int(ref[theirs]), "pass": int(k5[ours]) == int(ref[theirs])}
    checks["bb20_modularity"] = {"observed": float(k5.bb20_modularity), "e3_reference": float(ref.modularity),
                                 "pass": abs(float(k5.bb20_modularity) - float(ref.modularity)) < 1e-6}
gate_pass = bool(checks) and all(v["pass"] for v in checks.values())
C.write_json({"gate": "k=5 arm reproduces the E3 run on the same sample", "pass": gate_pass, "checks": checks},
             OUT / "validation.json")
print("E3 CROSS-CHECK GATE:", "PASS" if gate_pass else "FAIL")
for k, v in checks.items():
    print(f"  {k:18s} observed={v['observed']} e3_reference={v['e3_reference']} {'ok' if v['pass'] else 'MISMATCH'}")
assert gate_pass, "the k = 5 arm does not reproduce the E3 record for this sample"

E3 CROSS-CHECK GATE: PASS
  global_nodes       observed=10249 e3_reference=10249 ok
  global_edges       observed=13247 e3_reference=13247 ok
  global_components  observed=3775 e3_reference=3775 ok
  global_lcc_nodes   observed=5692 e3_reference=5692 ok
  bb20_nodes         observed=31 e3_reference=31 ok
  bb20_edges         observed=35 e3_reference=35 ok
  bb20_communities   observed=3 e3_reference=3 ok
  bb20_modularity    observed=0.34011616342443407 e3_reference=0.340116163424434 ok


In [6]:
# ============================================================
# COMPARISON BETWEEN ARMS, SUMMARY AND METADATA
# ============================================================
comparison = {}
for k in [k for k in arms if k != 5]:
    comp = {}
    for w in W_LEVELS:
        p5, pk = parts[5][w], parts[k][w]
        n5, nk = set(p5), set(pk)
        comp[f"bb{w}"] = {"nodes_k5": len(n5), f"nodes_k{k}": len(nk), "shared": len(n5 & nk),
                          "jaccard_nodes": len(n5 & nk) / max(1, len(n5 | nk)), **C.partition_agreement(p5, pk)}
    # Novelty of the extra keywords with respect to the document's k=5 set.
    exact, soft, new, total = 0, 0, 0, 0
    for d5, dk in zip(arms[5], arms[k]):
        s5 = set(d5)
        for t in dk:
            if not t or t == "null":
                continue
            total += 1
            if t in s5:
                exact += 1
            elif d5 and max(float(emb[t] @ emb[u]) for u in d5) >= C.TAU_SOFT:
                soft += 1
            else:
                new += 1
    comp["extra_keyword_novelty"] = {"keywords": total, "exact_restatement_of_k5": exact / max(1, total),
                                     "soft_restatement_of_k5": soft / max(1, total), "new_concept": new / max(1, total)}
    comparison[f"k5_vs_k{k}"] = comp
status = "complete" if len(arms) > 1 else "k=5 arm only; run run_extraction.py (needs OPENROUTER_API_KEY) to add other k"
C.write_json({"status": status, "arms": rows, "comparison": comparison, "w_levels": W_LEVELS}, OUT / "summary.json")
meta.update({"completed_utc": C.now_utc(), "timings_seconds": T.marks, "status": status})
C.write_json(meta, OUT / "metadata_analysis.json")

print("STATUS:", status)
if comparison:
    print(json.dumps(comparison, indent=2, default=C._json_default))
else:
    print("no comparison yet: only the k = 5 arm is available")
print("\nk = 5 arm on the 5,000-document sample:")
print(k5[["documents", "keywords_per_document_mean", "vocabulary", "null_tokens", "global_nodes", "global_edges",
          "global_components", "global_lcc_fraction", "bb20_nodes", "bb20_edges", "bb20_modularity", "bb20_communities",
          "bb5_nodes", "bb5_edges", "bb5_modularity", "bb5_communities"]].to_string())
print("\ntimings (s):", T.marks)
print("written:", sorted(p.name for p in OUT.iterdir()))

STATUS: k=5 arm only; run run_extraction.py (needs OPENROUTER_API_KEY) to add other k
no comparison yet: only the k = 5 arm is available

k = 5 arm on the 5,000-document sample:
documents                         5000
keywords_per_document_mean       4.995
vocabulary                       10249
null_tokens                         18
global_nodes                     10249
global_edges                     13247
global_components                 3775
global_lcc_fraction           0.555371
bb20_nodes                          31
bb20_edges                          35
bb20_modularity               0.340116
bb20_communities                     3
bb5_nodes                          184
bb5_edges                          244
bb5_modularity                0.370338
bb5_communities                      7

timings (s): {'sample': 4.314, 'dry_run_manifest': 5.185, 'embeddings': 26.138, 'arms': 27.338}
written: ['arms_comparison.csv', 'dry_run_manifest.json', 'metadata_analysis.json', 'sample_mapping.c

## Pending arm: k = 10

Not run (about 5,000 OpenRouter calls). To add it: `export OPENROUTER_API_KEY=...`, then `.venv/bin/python aditional_experiments/e4_k_sensitivity/run_extraction.py --k 10` from the repository root (resumable; `--dry-run` and `--max-docs N` available), and re-run this notebook.

## Check against the archived results

In [7]:
# ============================================================
# CHECK AGAINST THE ARCHIVED RESULTS (not reported in the manuscript)
# ============================================================
print("This experiment is an additional analysis; none of its numbers appears in main.tex.")
rows_chk = []


def add(section, quantity, a, b):
    if isinstance(b, (str, bool)) or isinstance(a, (str, bool)):
        ok = str(a) == str(b)
    elif isinstance(b, (int, np.integer)) and float(a).is_integer():
        ok = int(a) == int(b)
    else:
        ok = round(float(a), 6) == round(float(b), 6)
    rows_chk.append({"section": section, "quantity": quantity, "archived": a, "this run": b, "flag": "match" if ok else "differs"})


if "arms_comparison.csv" not in archived:
    print("no archived results were found before this run; nothing to compare")
else:
    old = archived["arms_comparison.csv"].set_index("arm")
    new = table.set_index("arm")
    for arm in new.index:
        if arm not in old.index:
            rows_chk.append({"section": "arm", "quantity": arm, "archived": "absent", "this run": "present", "flag": "new arm"})
            continue
        for m in [c for c in new.columns if not c.endswith("density")]:
            add(f"arm {arm}", m, old.loc[arm, m], new.loc[arm, m])
    if "validation.json" in archived:
        add("gate", "pass", archived["validation.json"]["pass"], gate_pass)
        for k, v in archived["validation.json"]["checks"].items():
            add("gate", f"{k} observed", v["observed"], checks[k]["observed"])
    if "dry_run_manifest.json" in archived:
        om = archived["dry_run_manifest.json"]
        for m in ("model", "k_values", "documents", "sample_sha256", "decoding", "retry_policy", "truncate_chars",
                  "records_truncated", "expected_calls", "wall_clock_lower_bound_hours", "estimated_cost_usd_at_list_price"):
            add("dry-run manifest", m, json.dumps(om.get(m)), json.dumps(manifest.get(m)))
        add("dry-run manifest", "system_message identical", True, om["system_message"] == manifest["system_message"])
        add("dry-run manifest", "prompt_template identical", True, om["prompt_template"] == manifest["prompt_template"])
    if "sample_mapping.csv" in archived:
        osm, nsm = archived["sample_mapping.csv"], mapping.drop(columns=["text_seen"])
        add("sample mapping", "rows", len(osm), len(nsm))
        for col in ("corpus_position", "source_row_abs", "EID_o_identificador", "insumo_row", "record_chars"):
            add("sample mapping", f"{col} identical", True, bool((osm[col].astype(str).values == nsm[col].astype(str).values).all()))
    if "summary.json" in archived:
        add("summary", "status", archived["summary.json"]["status"], status)
    check = pd.DataFrame(rows_chk)
    pd.set_option("display.width", 250)
    print(check.to_string(index=False))
    n_diff = int((check.flag == "differs").sum())
    print(f"\n{len(check)} comparisons: {len(check) - n_diff} match, {n_diff} differ")

This experiment is an additional analysis; none of its numbers appears in main.tex.
         section                         quantity                                                                      archived                                                                      this run  flag
         arm k=5                        documents                                                                          5000                                                                          5000 match
         arm k=5                   keywords_total                                                                         24975                                                                         24975 match
         arm k=5       keywords_per_document_mean                                                                         4.995                                                                         4.995 match
         arm k=5                       vocabulary                   